# Protocol 6 — iEEG All-or-None Ignition Dynamics

Simulates Pred 6.A–6.D from Protocol 6 (*Intracranial EEG During Near-Threshold Detection: All-or-None
Ignition Dynamics*), updated to **protocol specification v1.1.0**. Every prediction is
**confirmatory-exploratory (CE)** until an independent replication in ≥ 12 more patients.

* **Pred 6.A** — At a *fixed* near-threshold contrast, frontoparietal high-gamma (70–150 Hz) is bimodal:
  Hartigan dip p < 0.05, 2- vs 1-component Gaussian mixture ΔBIC > 10, and parametric-bootstrap LRT p < 0.05.
  *Falsified*: dip p > 0.20 **and** ΔBIC < 2 favoring one component. Intermediate outcomes are indeterminate.
* **Pred 6.B** — Intermediate-state dwell (25–75 % of the mixture-mean separation, episodes ≥ 100 ms) stays
  below 15 % of the 0–400 ms window. Regional specificity needs **positive** evidence for a one-component
  occipital model plus a direct frontoparietal-vs-occipital contrast. *Falsified*: dwell > 30 % or occipital bimodality.
* **Pred 6.C** — AC1 of the pre-stimulus high-gamma envelope (−500 to 0 ms) rises on detected trials
  (median participant Kendall τ ≥ 0.40). *Falsified*: τ < 0.20 with BF₀₁ ≥ 3; 0.20 ≤ τ < 0.40 is inconclusive.
  **Deferred pending estimator validation**: each 50 ms window holds only five samples at 100 Hz.
* **Pred 6.D** — Frontoparietal beta/gamma coherence (15–80 Hz) predicts seen vs unseen (point-biserial
  r > 0.40), peaks 200–400 ms post-stimulus, and is absent for the occipital control pair.
* **Pred 6.E** — Πⁱ modulation predicts ≥ 3 dissociable ignition-clustering timescales (BF₁₀ ≥ 100 vs single
  timescale; 1000 surrogates) in the aINS-covered subset. Described, not simulated.

> All data are synthetic. Inference in the registered study is single-subject hierarchical Bayesian
> (BF₁₀ ≥ 6 per prediction) with group BY-FDR q = 0.05.
>
> **EP-6 | Paper 2 | Clinical-empirical (CE) | Depends on: EP-2 | Spec v1.1.0, pre-collection | N target = 20, minimum = 12**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| Trials | 300 (150 near-threshold) | 8 blocks × 75 = 600: 150 catch, 150 suprathreshold, 300 near-threshold (75 at each of 0.02/0.04/0.08/0.16 Michelson) |
| Contrast grid | threshold-relative | **Absolute** Michelson grid [0, 0.02, 0.04, 0.08, 0.16, 0.32] (OSF governs) |
| Bimodality test | KS vs normal | Fixed-contrast dip + ΔBIC > 10 + parametric-bootstrap LRT |
| Pred 6.B | Region × bimodality p | Dwell < 15 % + positive one-component occipital evidence + direct contrast |
| Pred 6.C | τ > 0.3 | τ ≥ 0.40; inconclusive 0.20–0.40; deferred until AC1 estimator recovery is archived |
| Pred 6.D | Intermediate states > 100 ms | Frontoparietal coherence predicts seen/unseen (r > 0.40, 200–400 ms) |
| Pred 6.E | — | New: ≥ 3 clustering timescales in aINS-covered patients |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kendalltau, norm, pearsonr

try:
    import diptest  # optional: Hartigan's dip test

    HAVE_DIPTEST = True
except ImportError:
    HAVE_DIPTEST = False

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_6_ignition_ieeg.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

## 1 — Protocol 6 design (v1.1)

In [ ]:
BLOCKS, PIPE = PROTO["blocks"], PROTO["ieeg_analysis_pipeline"]
N_PATIENTS = PROTO["participants"]["n_target"]          # 20
CONTRASTS = BLOCKS["absolute_michelson_contrasts"]      # [0, 0.02, 0.04, 0.08, 0.16, 0.32]
N_PER_NEAR = BLOCKS["near_threshold_per_intermediate_contrast"]  # 75
FIXED_CONTRAST = 0.04                                    # one fixed near-threshold level for Pred 6.A
FS_ENV = PIPE["ac1_envelope_sample_rate_hz"]             # 100 Hz
AC1_WIN = PIPE["ac1_window_ms"]                          # 50 ms
AC1_EPOCH = PIPE["ac1_epoch_ms"]                         # [-500, 0]
N_WINDOWS = (AC1_EPOCH[1] - AC1_EPOCH[0]) // AC1_WIN    # 10
SAMPLES_PER_WIN = PIPE["ac1_samples_per_window"]        # 5
THR_A = PRED["Pred 6.A"]["confirming_thresholds"]
THR_B = PRED["Pred 6.B"]["confirming_thresholds"]
THR_C = PRED["Pred 6.C"]["confirming_thresholds"]
assert BLOCKS["catch_trials"] + BLOCKS["suprathreshold_trials"] + BLOCKS["near_threshold_trials"] \
    == BLOCKS["total_experimental_trials"]

print(f"{BLOCKS['experimental_blocks']} blocks × {BLOCKS['trials_per_block']} = "
      f"{BLOCKS['total_experimental_trials']} trials; contrasts {CONTRASTS}")
print(f"High-gamma {PIPE['high_gamma_hz']} Hz; AC1: {SAMPLES_PER_WIN} samples per {AC1_WIN} ms window at {FS_ENV} Hz")
print("Hartigan dip test:", "available" if HAVE_DIPTEST else "unavailable (pip install diptest) — mixture tests only")
rng = np.random.default_rng(9)

## 2 — Mixture tools: 1- vs 2-component Gaussian models and parametric-bootstrap LRT

In [ ]:
def _pdf(x, mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))


def fit_gmm2(x, n_iter=200, tol=1e-6):
    """EM for a two-component 1-D Gaussian mixture; returns (log-likelihood, params)."""
    w, mu, sd = 0.5, np.percentile(x, [25, 75]), np.full(2, x.std())
    ll_old = -np.inf
    for _ in range(n_iter):
        p1, p2 = w * _pdf(x, mu[0], sd[0]), (1 - w) * _pdf(x, mu[1], sd[1])
        ll = np.sum(np.log(p1 + p2 + 1e-300))
        if ll - ll_old < tol:
            break
        ll_old = ll
        g = p1 / (p1 + p2 + 1e-300)
        w = np.clip(g.mean(), 1e-3, 1 - 1e-3)
        mu = np.array([np.sum(g * x) / g.sum(), np.sum((1 - g) * x) / (1 - g).sum()])
        sd = np.sqrt([np.sum(g * (x - mu[0]) ** 2) / g.sum(), np.sum((1 - g) * (x - mu[1]) ** 2) / (1 - g).sum()])
        sd = np.maximum(sd, 1e-3)
    return ll, (w, mu, sd)


def mixture_evidence(x, n_boot=99, rng=None):
    """ΔBIC (1-comp − 2-comp; positive favors two) and parametric-bootstrap LRT p."""
    n = len(x)
    ll1 = np.sum(norm.logpdf(x, x.mean(), x.std()))
    ll2, params = fit_gmm2(x)
    delta_bic = (-2 * ll1 + 2 * np.log(n)) - (-2 * ll2 + 5 * np.log(n))
    lrt = 2 * (ll2 - ll1)
    rng = rng or np.random.default_rng(0)
    null = []
    for _ in range(n_boot):
        xb = rng.normal(x.mean(), x.std(), n)
        null.append(2 * (fit_gmm2(xb, 100)[0] - np.sum(norm.logpdf(xb, xb.mean(), xb.std()))))
    p_boot = (np.sum(np.array(null) >= lrt) + 1) / (n_boot + 1)
    return delta_bic, p_boot, params


def dip_p(x):
    return diptest.diptest(x)[1] if HAVE_DIPTEST else np.nan


def by_fdr(pvals, q=0.05):
    """Benjamini–Yekutieli step-up; returns boolean rejections."""
    p = np.asarray(pvals)
    m = len(p)
    c_m = np.sum(1 / np.arange(1, m + 1))
    order = np.argsort(p)
    ok = p[order] <= q * np.arange(1, m + 1) / (m * c_m)
    k = np.max(np.flatnonzero(ok)) + 1 if ok.any() else 0
    rej = np.zeros(m, bool)
    rej[order[:k]] = True
    return rej

## 3 — Pred 6.A / 6.B: Fixed-contrast bimodality and regional specificity

In [ ]:
patients = []
for pid in range(N_PATIENTS):
    p_seen = rng.uniform(0.35, 0.65)                       # detection rate at the fixed contrast
    seen = rng.random(N_PER_NEAR) < p_seen
    hg_fp = np.where(seen, rng.normal(0.72, 0.08, N_PER_NEAR), rng.normal(0.22, 0.07, N_PER_NEAR))
    hg_occ = rng.normal(0.45, 0.15, N_PER_NEAR) + 0.05 * seen  # graded occipital response
    dbic_fp, pb_fp, params_fp = mixture_evidence(hg_fp, rng=rng)
    dbic_occ, pb_occ, _ = mixture_evidence(hg_occ, rng=rng)
    patients.append(dict(seen=seen, hg_fp=hg_fp, hg_occ=hg_occ, dbic_fp=dbic_fp, pb_fp=pb_fp,
                         dbic_occ=dbic_occ, pb_occ=pb_occ, params_fp=params_fp,
                         dip_fp=dip_p(hg_fp), dip_occ=dip_p(hg_occ)))

rej = by_fdr([p["pb_fp"] for p in patients], q=PROTO["power_analysis"]["group_by_fdr_q"])
n_bimodal = sum(p["dbic_fp"] > THR_A["delta_bic_strict_minimum"] and r for p, r in zip(patients, rej))
n_occ_one = sum(p["dbic_occ"] < -2 for p in patients)  # positive evidence for one component (ΔBIC < −2)
print(f"Frontoparietal: ΔBIC > 10 and BY-FDR-significant bootstrap LRT in {n_bimodal}/{N_PATIENTS} patients "
      f"(median ΔBIC = {np.median([p['dbic_fp'] for p in patients]):.1f})")
print(f"Occipital: positive one-component evidence (ΔBIC < −2) in {n_occ_one}/{N_PATIENTS} patients "
      f"(median ΔBIC = {np.median([p['dbic_occ'] for p in patients]):.1f})")
if HAVE_DIPTEST:
    print(f"Median dip p: frontoparietal {np.median([p['dip_fp'] for p in patients]):.3f}, "
          f"occipital {np.median([p['dip_occ'] for p in patients]):.3f}")
confirmed_6A = n_bimodal >= N_PATIENTS / 2 and (not HAVE_DIPTEST or np.median([p["dip_fp"] for p in patients]) < 0.05)
print(f"Pred 6.A → {verdict(confirmed_6A)}" + ("" if HAVE_DIPTEST else "  [dip conjunct not evaluated]"))

In [ ]:
# Pred 6.B: intermediate-state dwell in the 0–400 ms post-stimulus window (100 Hz envelope)
t_post = np.arange(THR_B["window_ms"][0], THR_B["window_ms"][1], 1000 // FS_ENV)
lo_frac, hi_frac = THR_B["intermode_fraction_bounds"]
min_samples = THR_B["episode_duration_ms_minimum"] // (1000 // FS_ENV)


def dwell_fraction(trace, low, high):
    frac = (trace - low) / (high - low)
    inter = (frac > lo_frac) & (frac < hi_frac)
    total, run = 0, 0
    for v in np.append(inter, False):
        if v:
            run += 1
        else:
            total += run if run >= min_samples else 0
            run = 0
    return total / len(trace)


dwell = []
for p in patients:
    _, mu, _ = p["params_fp"]
    low, high = np.sort(mu)
    for s in p["seen"]:
        onset = rng.uniform(150, 300)
        tau = rng.uniform(8, 20) if rng.random() > 0.08 else rng.uniform(60, 120)  # occasional slow transition
        trace = low + (high - low) * (1 / (1 + np.exp(-(t_post - onset) / tau))) * s + rng.normal(0, 0.03, len(t_post))
        dwell.append(dwell_fraction(trace, low, high))
dwell = np.array(dwell)
confirmed_6B = dwell.mean() < THR_B["dwell_fraction_strict_maximum"] and n_occ_one >= N_PATIENTS / 2
falsified_6B = dwell.mean() > PRED["Pred 6.B"]["falsifying_thresholds"]["dwell_fraction_strict_minimum"]
print(f"Mean intermediate dwell = {dwell.mean():.1%} of 0–400 ms (criterion < 15%; falsifier > 30%)")
print(f"Pred 6.B → {verdict(confirmed_6B, falsified_6B)}  (direct regional contrast awaits the locked manifest)")

## 4 — Pred 6.C: Why the AC1 test is deferred

The registered pipeline estimates lag-1 autocorrelation from **five samples** per 50 ms window. The
recovery check below draws AR(1) envelopes with known φ and estimates AC1 from five samples. Single-window
estimates are strongly biased toward negative values, compressed (slope well below 1), and very noisy.
Averaging over trials can still recover a monotone *trend*, but absolute AC1 values cannot be interpreted.
That is why v1.1 defers Pred 6.C until estimator recovery and group-median τ power are archived.

In [ ]:
def ac1_rows(X):
    X = X - X.mean(1, keepdims=True)
    return np.sum(X[:, 1:] * X[:, :-1], 1) / np.sum(X * X, 1)


def ar1_draws(phi, n, size, rng):
    """`size` stationary AR(1) series of length n with unit innovation variance."""
    y = np.empty((size, n))
    y[:, 0] = rng.normal(0, 1 / np.sqrt(1 - phi**2), size)
    e = rng.normal(0, 1, (size, n))
    for i in range(1, n):
        y[:, i] = phi * y[:, i - 1] + e[:, i]
    return y


phis = np.array([0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
est = np.array([ac1_rows(ar1_draws(f, SAMPLES_PER_WIN, 2000, rng)) for f in phis])
for f, e in zip(phis, est):
    print(f"  true φ = {f:.1f} → mean AC1 estimate {e.mean():+.3f}  (SD {e.std():.3f})")
print(f"Recovered slope dÂC1/dφ = {np.polyfit(phis, est.mean(1), 1)[0]:.2f} (ideal 1.00)")

In [ ]:
# Participant-level trend on trial-averaged AC1 across the 10 pre-stimulus windows
phi_detected = np.linspace(0.20, 0.60, N_WINDOWS)  # critical slowing on subsequently detected trials
phi_missed = np.full(N_WINDOWS, 0.25)
tau_det, tau_miss = [], []
for p in patients:
    n_det, n_miss = p["seen"].sum(), (~p["seen"]).sum()
    a_det = [ac1_rows(ar1_draws(f, SAMPLES_PER_WIN, n_det, rng)).mean() for f in phi_detected]
    a_miss = [ac1_rows(ar1_draws(f, SAMPLES_PER_WIN, n_miss, rng)).mean() for f in phi_missed]
    tau_det.append(kendalltau(np.arange(N_WINDOWS), a_det)[0])
    tau_miss.append(kendalltau(np.arange(N_WINDOWS), a_miss)[0])
tau_med = np.median(tau_det)
status = PRED["Pred 6.C"]["evaluation_status"]
print(f"Median participant τ: detected = {tau_med:.2f}, non-detected = {np.median(tau_miss):.2f}")
print(f"Pred 6.C → NOT EVALUATED ({status}); synthetic τ shown for estimator illustration only")

## 5 — Pred 6.D: Frontoparietal coherence predicts seen vs unseen

In [ ]:
t_coh = np.arange(0, 600, 10)
r_fp, r_occ, peak_ms = [], [], []
for p in patients:
    seen = p["seen"].astype(float)
    bump = np.exp(-0.5 * ((t_coh - 300) / 60) ** 2)  # coherence increase peaking at 300 ms on seen trials
    trial_offset = rng.normal(0, 0.06, (len(seen), 1))  # slow trial-to-trial coherence fluctuation
    coh = 0.30 + trial_offset + 0.12 * np.outer(seen, bump) + rng.normal(0, 0.06, (len(seen), len(t_coh)))
    coh_occ = 0.30 + trial_offset + rng.normal(0, 0.06, (len(seen), len(t_coh)))
    win = (t_coh >= 200) & (t_coh <= 400)
    r_fp.append(pearsonr(coh[:, win].mean(1), seen)[0])
    r_occ.append(pearsonr(coh_occ[:, win].mean(1), seen)[0])
    peak_ms.append(t_coh[np.argmax(coh[seen == 1].mean(0) - coh[seen == 0].mean(0))])
r_fp_med, r_occ_med, peak_med = np.median(r_fp), np.median(r_occ), np.median(peak_ms)
confirmed_6D = r_fp_med > 0.40 and 200 <= peak_med <= 400 and abs(r_occ_med) < 0.20
falsified_6D = r_fp_med < 0.20 or not (200 <= peak_med <= 400)
print(f"Median point-biserial r: frontoparietal = {r_fp_med:.2f}, occipital control = {r_occ_med:.2f}; "
      f"peak = {peak_med:.0f} ms")
print(f"Pred 6.D → {verdict(confirmed_6D, falsified_6D)}")
print("Pred 6.E scope:", PRED["Pred 6.E"]["scope"])

## 6 — Visualize Pred 6.A–6.D

In [ ]:
ex = patients[0]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

ax = axes[0]
ax.hist(ex["hg_fp"], bins=20, color="#2166ac", alpha=0.75, edgecolor="white", density=True, label="Frontoparietal")
ax.hist(ex["hg_occ"], bins=20, color="#AAAAAA", alpha=0.65, edgecolor="white", density=True, label="Occipital")
ax.set_title(f"Pred 6.A/6.B — fixed contrast {FIXED_CONTRAST}\nΔBIC FP = {ex['dbic_fp']:.0f}, occ = {ex['dbic_occ']:.0f}",
             fontsize=10)
ax.set_xlabel("High-gamma (a.u.)")
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.hist(dwell * 100, bins=20, color="#9966FF", alpha=0.85, edgecolor="white")
ax.axvline(15, ls="--", color="k", lw=1, label="15% criterion")
ax.axvline(30, ls=":", color="#d6604d", lw=1.2, label="30% falsifier")
ax.set_xlabel("Intermediate dwell (% of 0–400 ms)")
ax.set_ylabel("Seen trials")
ax.set_title(f"Pred 6.B — mean dwell {dwell.mean():.1%}", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
ax.plot(phis, phis, "k--", lw=1, label="ideal recovery")
ax.errorbar(phis, est.mean(1), yerr=est.std(1), fmt="o-", color="#d6604d", capsize=3,
            label=f"{SAMPLES_PER_WIN}-sample estimate")
ax.set_xlabel("True AC1 (φ)")
ax.set_ylabel("Estimated AC1")
ax.set_title("Pred 6.C — AC1 estimator (deferred)", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[3]
ax.hist(r_fp, bins=10, color="#2166ac", alpha=0.75, edgecolor="white", label="Frontoparietal")
ax.hist(r_occ, bins=10, color="#AAAAAA", alpha=0.75, edgecolor="white", label="Occipital control")
ax.axvline(0.40, ls="--", color="k", lw=1, label="r = 0.40")
ax.set_xlabel("Point-biserial r (coherence, seen)")
ax.set_ylabel("Patients")
ax.set_title("Pred 6.D — coherence 200–400 ms", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Protocol 6 — iEEG ignition dynamics (CE) · spec v1.1.0 · synthetic N = {N_PATIENTS}", fontsize=12)
plt.tight_layout()
plt.show()

## 7 — Why Pred 6.C matters: APGI vs. Global Workspace Theory

| Prediction | Standard GWT | APGI |
|---|---|---|
| Bistable high-gamma at fixed near-threshold contrast (6.A) | Yes | Yes |
| Brief intermediate dwell, frontoparietal-specific (6.B) | Yes | Yes |
| **Pre-stimulus AC1 increase / critical slowing (6.C)** | **No** | **Yes** |
| Frontoparietal coherence 200–400 ms (6.D) | Yes | Yes |
| ≥ 3 Πⁱ-modulated clustering timescales (6.E) | No (single timescale) | Yes |

Pred 6.C is the bifurcation criterion. v1.1 makes no bifurcation claim until AC1 estimator recovery and
power are archived. The recovery check above shows why: a five-sample AC1 compresses and biases the true
autocorrelation, so the trend test depends on trial counts and on the slope of that bias. A failed 6.C
would remove the bifurcation interpretation without falsifying threshold crossing as such.